# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR^2 dataset using the `mlcroissant` library, referencing all entities with their `@id` as per the Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Croissant metadata is accessed as an object, not a dict
metadata = dataset.metadata
print(f"Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Version: {metadata.version}")
print(f"Identifier: {metadata.identifier}")

## 2. Data Overview
Review available record sets, fields, and their IDs according to the Croissant schema.

Each entity (record set, field, column) should be referenced by its `@id`.

In [ ]:
# List all record sets and their IDs
record_sets = dataset.record_sets()
print("Record Sets and their @id:")
for rs in record_sets:
    print(f"- Name: {rs.name}, @id: {rs.id}")

# For each record set, list fields and columns by @id
overview = {}
for rs in record_sets:
    print(f"\nRecordSet: {rs.name} (@id: {rs.id})")
    fields = rs.fields()
    overview[rs.id] = {'fields': [], 'columns': []}
    for field in fields:
        print(f"  Field: {field.name}, @id: {field.id}, dtype: {field.data_type}")
        overview[rs.id]['fields'].append(field.id)
        for col in field.columns():
            print(f"    Column: {col.name}, @id: {col.id}, dtype: {col.data_type}")
            overview[rs.id]['columns'].append(col.id)


## 3. Data Extraction
Load data from each record set into a DataFrame for analysis, referencing record set and field `@id`s.

In [ ]:
# Create a list of record set @ids for extraction
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

# Load each record set records as DataFrame
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"\nLoaded record set: {rs_id}\nColumns: {df.columns.tolist()}")

# Choose the first record set for demonstration
demo_rs_id = record_set_ids[0] if record_set_ids else None
if demo_rs_id:
    print(f"\nPreview of {demo_rs_id}")
    display(dataframes[demo_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Here we process the records—filter values, normalize numeric columns, and group/categorize records referencing fields by their `@id`.

We will select a numeric field, remove outliers, normalize values, and group by a relevant categorical field.

In [ ]:
# EDA on a numeric field for the demo record set
if demo_rs_id is not None:
    df = dataframes[demo_rs_id]
    # Find numeric fields using overview (dtype includes 'Integer', 'Float', 'Number')
    demo_fields = dataset.get_record_set(demo_rs_id).fields()
    numeric_field_ids = [f.id for f in demo_fields if f.data_type in ('schema:Integer', 'schema:Float', 'schema:Number')]
    print(f"Numeric field candidates (@id): {numeric_field_ids}")
    
    # Use the first numeric field found
    if numeric_field_ids:
        numeric_field = numeric_field_ids[0]
        if numeric_field in df.columns:
            print(f"Using numeric field @id: {numeric_field}")
            threshold = df[numeric_field].mean() if np.issubdtype(df[numeric_field].dtype, np.number) else 10
            filtered_df = df[df[numeric_field] > threshold]
            print(f"Filtered records with {numeric_field} > {threshold}:")
            print(filtered_df.head())

            # Normalize
            if filtered_df[numeric_field].std() != 0:
                filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
                print(f"Normalized {numeric_field} for filtered records:")
                print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

            # Group by a categorical field (@id)
            group_candidate_ids = [f.id for f in demo_fields if f.data_type == 'schema:Text' or f.data_type == 'schema:Boolean']
            if group_candidate_ids:
                group_field = group_candidate_ids[0]
                print(f"Grouping by: {group_field}")
                if group_field in filtered_df.columns:
                    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
                    print("Grouped means:")
                    print(grouped_df.head())
        else:
            print("No numeric field found in columns for EDA.")
    else:
        print("No numeric fields detected for EDA.")

## 5. Visualization
Visualize distributions or relationships using matplotlib—referencing fields and columns by their `@id`.

In [ ]:
# Basic visualization for numeric fields
if demo_rs_id and numeric_field_ids:
    numeric_field = numeric_field_ids[0]
    df = dataframes[demo_rs_id]
    if numeric_field in df.columns:
        plt.figure(figsize=(8,4))
        plt.hist(df[numeric_field].dropna(), bins=10, color='skyblue', edgecolor='black')
        plt.title(f"Distribution of {numeric_field} (@id)")
        plt.xlabel(numeric_field)
        plt.ylabel('Frequency')
        plt.show()

    # Visualize grouping if available
    if group_candidate_ids and group_candidate_ids[0] in df.columns:
        group_field = group_candidate_ids[0]
        group_means = df.groupby(group_field)[numeric_field].mean()
        plt.figure(figsize=(8,4))
        group_means.plot(kind='bar', color='salmon')
        plt.title(f"Mean {numeric_field} (@id) by {group_field} (@id)")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.show()

## 6. Conclusion
This notebook illustrated loading metadata, extracting record sets, referencing all entities by their `@id`, and processing and visualizing the FAIR^2 dataset using `mlcroissant`.

Key findings:
- The Croissant schema enables entity referencing with `@id`, ensuring traceability.
- The dataset contains clinicopathological variables for 77 colorectal cancer survivors.
- Numeric and categorical fields are accessible for filtering and group analysis.
- Visualization offers insight into value distributions and relationships, supporting downstream ML or clinical research.